# Bayesian classification

In [43]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True, as_frame=False)
X[:5], y[:5]

(array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
         3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
         8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
         3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
         1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01],
        [2.057e+01, 1.777e+01, 1.329e+02, 1.326e+03, 8.474e-02, 7.864e-02,
         8.690e-02, 7.017e-02, 1.812e-01, 5.667e-02, 5.435e-01, 7.339e-01,
         3.398e+00, 7.408e+01, 5.225e-03, 1.308e-02, 1.860e-02, 1.340e-02,
         1.389e-02, 3.532e-03, 2.499e+01, 2.341e+01, 1.588e+02, 1.956e+03,
         1.238e-01, 1.866e-01, 2.416e-01, 1.860e-01, 2.750e-01, 8.902e-02],
        [1.969e+01, 2.125e+01, 1.300e+02, 1.203e+03, 1.096e-01, 1.599e-01,
         1.974e-01, 1.279e-01, 2.069e-01, 5.999e-02, 7.456e-01, 7.869e-01,
         4.585e+00, 9.403e+01, 6.150e-03, 4.006e-02, 3.832e-02, 2.058e-02,
         2.250e-02, 4.5

In [44]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

## Model

Bernoulli likelihood with a sigmoid link 

$$
y \in \{0, 1\}
$$
$$
p(y = 1 \mid x, w) = \sigma(x^\top w) ,\quad \sigma(z) = \frac{1}{1 + e^{-z}}
$$
$$
p(y \mid X, w) = \prod_{i=1}^n p(y_i \mid x_i, w)^{y_i} (1 - p(y_i \mid x_i, w))^{1 - y_i} = \prod_{i=1}^n \sigma(x_i ^\top w)^{y_i} (1 -  \sigma(x_i ^\top w))^{1 - y_i}
$$
The Bernoulli distribution is a discrete probability distribution that models a binary outcome for one trial.

---

Assume prior on weights
$$
w \sim \mathcal{N}(0, \alpha^2I)
$$
with precision parameter $\alpha$. 

Since the likelihood is not Gaussian, the posterior is not a conjugate Gaussian, and there is no closed form for the posterior. However, common is to use Laplace approximation for the posterior as Gaussian
$$
p(w \mid X, y) \approx \mathcal{N}(\mu, \Sigma)
$$

---

The first step is to find the MAP estimate of the mean
$$
\mu = \arg \max_w \log p(w \mid X, y)
$$
Equvalent to minimizing
$$
\mathcal{L}(w) = -\sum_{i=1}^N \left( y_i \log \sigma(x_i ^\top w) + (1 - y_i) \log (1 - \sigma(x_i ^\top w))  \right) + \frac{\alpha}{2} w^\top w
$$
The covariance is estimated from the Hessian
$$
\Sigma = H^{-1} = (\alpha I + X^\top R X)^{-1}
$$
where
$$
R = diag (\sigma_i(1 - \sigma_i)), \quad \sigma_i = (x_i ^\top \mu)
$$

---

The exact predictive distribution is 
$$
p(y_* = 1 \mid x_*, X, y) = \int \sigma(x_*^\top w) p(w \mid X, y) \; dw
$$
which is intractable, and approximated by
$$
\mu_* = x_* ^\top \mu
$$
$$
\sigma_* ^2 = x_* ^\top \Sigma x_*
$$
using the probit approximateion
$$
p(y_* = 1 \mid x_*) \approx \sigma \left( \frac{\mu_*}{\sqrt{1 + \frac{\pi}{8}\sigma_*^2}} \right)
$$

In [ ]:
import numpy as np
from scipy.special import expit  # sigmoid
from scipy.optimize import minimize


class BayesianLogisticRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.mu = None
        self.Sigma = None

    def _neg_log_posterior(self, w, X, y):
        z = X @ w
        log_likelihood = np.sum(
            y * np.log(expit(z) + 1e-9) +
            (1 - y) * np.log(1 - expit(z) + 1e-9)
        )
        log_prior = -0.5 * self.alpha * np.dot(w, w)
        return -(log_likelihood + log_prior)

    def _grad(self, w, X, y):
        z = X @ w
        p = expit(z)
        grad_likelihood = X.T @ (p - y)
        grad_prior = self.alpha * w
        return grad_likelihood + grad_prior

    def fit(self, X, y):
        y = y.ravel()   
        
        N, D = X.shape
        w0 = np.zeros(D)

        res = minimize(
            self._neg_log_posterior,
            w0,
            args=(X, y),
            jac=self._grad,
            method='BFGS'
        )

        self.mu = res.x

        z = X @ self.mu
        p = expit(z)
        R = p * (1 - p)

        XR = X * R[:, np.newaxis]
        H = self.alpha * np.eye(D) + X.T @ XR

        self.Sigma = np.linalg.inv(H)

    def predict_proba(self, X_star):
        mu_star = X_star @ self.mu

        # Variance term
        var_star = np.sum(X_star @ self.Sigma * X_star, axis=1)

        # Approximate predictive probability
        kappa = np.sqrt(1 + (np.pi / 8) * var_star)
        probs = expit(mu_star / kappa)

        return probs

    def predict(self, X_star, threshold=0.5):
        return (self.predict_proba(X_star) >= threshold).astype(int)
    
    def predict_proba_with_uncertainty(self, X_star, n_samples=1000):
        """
        Returns:
            mean probability
            std (uncertainty)
            95% confidence interval
        """
        # Sample from posterior
        w_samples = np.random.multivariate_normal(
            self.mu, self.Sigma, size=n_samples
        )  # (S, D)

        # Compute logits
        logits = X_star @ w_samples.T   # (N, S)

        # Apply sigmoid
        probs = 1 / (1 + np.exp(-logits))

        # Aggregate
        mean = probs.mean(axis=1)
        std = probs.std(axis=1)

        # 95% confidence interval
        lower = np.percentile(probs, 2.5, axis=1)
        upper = np.percentile(probs, 97.5, axis=1)

        return mean, std, np.vstack((lower, upper)).T

In [46]:
# Fit model
model = BayesianLogisticRegression(alpha=1.0)
model.fit(X, y)

# Standard deviation of weights
weight_std = np.sqrt(np.diag(model.Sigma))

# Confidence intervals for weights
ci_lower = model.mu - 1.96 * weight_std
ci_upper = model.mu + 1.96 * weight_std

print("Weight means:", model.mu[:5])
print("Weight std devs:", weight_std[:5])

#print("95% CI for weights:")
#for i in range(len(model.mu)):
#    print(f"w[{i}]: [{ci_lower[i]:.3f}, {ci_upper[i]:.3f}]")

Weight means: [-0.30637854 -0.37595905 -0.29907509 -0.47415076 -0.12480353]
Weight std devs: [0.88412463 0.54081532 0.89545464 0.90830528 0.60757727]


In [47]:
# Predict
probs = model.predict_proba(X)
preds = model.predict(X)

print("Probabilities:", probs[:5])
print("Predictions:", preds[:5])

Probabilities: [4.62723203e-05 8.66424252e-04 5.86683294e-05 1.13663800e-02
 5.03022400e-04]
Predictions: [0 0 0 0 0]


To get uncertainty (confidence) about predictions in Bayesian logistic regression, you need to account for the fact that the weights www are uncertain, not fixed.

Monte Carlo predictive uncertainty sample weights
$$
w^{(s)} \sim \mathcal{N}(\mu, \Sigma)
$$
to predict uncertainty
$$
p^{(s)} = \sigma(x_*^\top w^{(s)})
$$
and then aggregate $p^{(s)}$ (mean, std). 

In [48]:
mean, std, ci = model.predict_proba_with_uncertainty(X)

# For each prediction:
# Mean → predicted probability
# Std → confidence (high = uncertain)
# CI → range of plausible probabilities
print("Mean probability:", mean[:5])
print("Uncertainty (std):", std[:5])
print("95% CI:", ci[:5])

Mean probability: [2.87374473e-08 9.49626442e-05 7.89615626e-07 5.44596284e-03
 7.10450929e-05]
Uncertainty (std): [2.68595482e-07 3.21408058e-04 2.55170999e-06 2.91307929e-02
 2.20749839e-04]
95% CI: [[1.48378051e-12 1.69286339e-07]
 [6.56628881e-07 7.23107398e-04]
 [2.65310182e-09 6.32643799e-06]
 [3.57662745e-06 3.66052197e-02]
 [9.47776882e-07 5.54796951e-04]]


In [49]:
# Epistemic uncertainty is captured by the std and CI. 
# High std means the model is uncertain about the prediction, often due to lack of data 
# in that region of feature space.